In [2]:
from pathlib import Path

import yaml

from reader import read_csv

project = Path(r"C:\tap-python-git")

with open(project / "config.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

data_folder = project / config["paths"]["data_dir"]

datasets = {}

for filename in config["files"]:
    file_path = data_folder / filename
    records = read_csv(str(file_path))
    datasets[filename] = records

    print(f"{filename}: {len(records):,} records")

print("-" * 60)
print("Total files loaded:", len(datasets))
print("Total records loaded:", sum(len(rows) for rows in datasets.values()))

olist_customers_dataset.csv: 500 records
olist_geolocation_dataset.csv: 2,229 records
olist_orders_dataset.csv: 500 records
olist_order_items_dataset.csv: 576 records
olist_order_payments_dataset.csv: 545 records
olist_order_reviews_dataset.csv: 489 records
olist_products_dataset.csv: 512 records
olist_sellers_dataset.csv: 317 records
product_category_name_translation.csv: 71 records
------------------------------------------------------------
Total files loaded: 9
Total records loaded: 5739


In [3]:
column_mappings = config["column_mappings"]

def apply_mapping(record, mappings):
    return {
        mappings.get(column, column): value
        for column, value in record.items()
    }

transformed_datasets = {}

for filename, records in datasets.items():
    transformed_datasets[filename] = [
        apply_mapping(record, column_mappings)
        for record in records
    ]

    print(f"Mapping completed: {filename}")

print("Total transformed datasets:", len(transformed_datasets))

Mapping completed: olist_customers_dataset.csv
Mapping completed: olist_geolocation_dataset.csv
Mapping completed: olist_orders_dataset.csv
Mapping completed: olist_order_items_dataset.csv
Mapping completed: olist_order_payments_dataset.csv
Mapping completed: olist_order_reviews_dataset.csv
Mapping completed: olist_products_dataset.csv
Mapping completed: olist_sellers_dataset.csv
Mapping completed: product_category_name_translation.csv
Total transformed datasets: 9


In [4]:
filename = "olist_order_items_dataset.csv"

print("Before transformation:")
print(list(datasets[filename][0].keys()))

print("\nAfter transformation:")
print(list(transformed_datasets[filename][0].keys()))

assert "item_price" in transformed_datasets[filename][0]
assert "shipping_fee" in transformed_datasets[filename][0]
assert "price" not in transformed_datasets[filename][0]

print("\nColumn mapping test PASSED")

Before transformation:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

After transformation:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'item_price', 'shipping_fee']

Column mapping test PASSED


In [5]:
import csv
import json

from day14_pipeline import REQUIRED_FIELDS, validate_row

valid_datasets = {}
bad_records = []
validation_summary = []

for filename, records in datasets.items():
    valid_records = []
    rejected_count = 0

    for row_number, record in enumerate(records, start=1):
        try:
            clean_record = validate_row(
                record.copy(),
                REQUIRED_FIELDS[filename],
                config
            )

            valid_records.append(clean_record)

        except (ValueError, TypeError) as error:
            rejected_count += 1

            bad_records.append({
                "source_file": filename,
                "record_number": row_number,
                "rejection_reason": str(error),
                "raw_record": json.dumps(record, ensure_ascii=False)
            })

    valid_datasets[filename] = valid_records

    rejection_rate = rejected_count / len(records) if records else 0

    validation_summary.append({
        "file": filename,
        "total": len(records),
        "valid": len(valid_records),
        "rejected": rejected_count,
        "rejection_rate": round(rejection_rate * 100, 2)
    })

    print(
        f"{filename} | "
        f"Total: {len(records)} | "
        f"Valid: {len(valid_records)} | "
        f"Rejected: {rejected_count}"
    )

olist_customers_dataset.csv | Total: 500 | Valid: 500 | Rejected: 0
olist_geolocation_dataset.csv | Total: 2229 | Valid: 2229 | Rejected: 0
olist_orders_dataset.csv | Total: 500 | Valid: 500 | Rejected: 0
olist_order_items_dataset.csv | Total: 576 | Valid: 576 | Rejected: 0
olist_order_payments_dataset.csv | Total: 545 | Valid: 545 | Rejected: 0
olist_order_reviews_dataset.csv | Total: 489 | Valid: 489 | Rejected: 0
olist_products_dataset.csv | Total: 512 | Valid: 512 | Rejected: 0
olist_sellers_dataset.csv | Total: 317 | Valid: 317 | Rejected: 0
product_category_name_translation.csv | Total: 71 | Valid: 71 | Rejected: 0


In [6]:
output_folder = project / config["paths"]["output_dir"]
output_folder.mkdir(exist_ok=True)

bad_records_path = output_folder / "bad_records.csv"

with open(bad_records_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "source_file",
            "record_number",
            "rejection_reason",
            "raw_record"
        ]
    )

    writer.writeheader()
    writer.writerows(bad_records)

print("Total rejected records:", len(bad_records))
print("Bad records saved:", bad_records_path)

Total rejected records: 0
Bad records saved: C:\tap-python-git\output\bad_records.csv


In [7]:
total_records = sum(len(rows) for rows in datasets.values())
total_valid = sum(len(rows) for rows in valid_datasets.values())
total_rejected = len(bad_records)

assert total_records == total_valid + total_rejected

print("Total records:", total_records)
print("Valid records:", total_valid)
print("Rejected records:", total_rejected)
print("Reconciliation test PASSED")

for result in validation_summary:
    if result["rejection_rate"] > config["thresholds"]["max_rejection_rate"] * 100:
        print("Threshold exceeded:", result["file"])

Total records: 5739
Valid records: 5739
Rejected records: 0
Reconciliation test PASSED


In [10]:
from day15error_module import (
    SchemaError,
    FileFormatError,
    DataQualityError,
    check_source,
    check_quality
)

successful_files = []
failed_files = {}

threshold = config["thresholds"]["max_rejection_rate"]

for result in validation_summary:
    filename = result["file"]

    try:
        check_source(
            data_folder / filename,
            REQUIRED_FIELDS[filename]
        )

        check_quality(
            result["total"],
            result["rejected"],
            threshold
        )

        successful_files.append(filename)
        print(f"SUCCESS: {filename}")

    except (SchemaError, FileFormatError, DataQualityError) as error:
        failed_files[filename] = str(error)
        print(f"FAILED: {filename} | {error}")

    finally:
        print(f"Check completed: {filename}")

print("\nSuccessful files:", len(successful_files))
print("Failed files:", len(failed_files))

pipeline_status = "FAILED" if failed_files else "SUCCESS"

print("Pipeline status:", pipeline_status)

SUCCESS: olist_customers_dataset.csv
Check completed: olist_customers_dataset.csv
SUCCESS: olist_geolocation_dataset.csv
Check completed: olist_geolocation_dataset.csv
SUCCESS: olist_orders_dataset.csv
Check completed: olist_orders_dataset.csv
SUCCESS: olist_order_items_dataset.csv
Check completed: olist_order_items_dataset.csv
SUCCESS: olist_order_payments_dataset.csv
Check completed: olist_order_payments_dataset.csv
SUCCESS: olist_order_reviews_dataset.csv
Check completed: olist_order_reviews_dataset.csv
SUCCESS: olist_products_dataset.csv
Check completed: olist_products_dataset.csv
SUCCESS: olist_sellers_dataset.csv
Check completed: olist_sellers_dataset.csv
SUCCESS: product_category_name_translation.csv
Check completed: product_category_name_translation.csv

Successful files: 9
Failed files: 0
Pipeline status: SUCCESS


In [11]:
try:
    check_quality(100, 30, threshold)

except DataQualityError as error:
    print("Exception handled:", error)

finally:
    print("Pipeline execution continues")

Exception handled: Rejection rate 30.00% exceeds threshold 20.00%
Pipeline execution continues


In [12]:
try:
    check_source(
        data_folder / "non_existing_file.csv",
        ["order_id"]
    )

except FileFormatError as error:
    print("Exception handled:", error)

print("Graceful failure test completed")

Exception handled: Missing file: non_existing_file.csv
Graceful failure test completed


In [15]:
from day15_logging import setup_logger

log_folder = project / config["paths"]["log_dir"]

logger, log_path = setup_logger(log_folder)

try:
    logger.info("Day 15 pipeline validation report started")

    for result in validation_summary:
        filename = result["file"]

        logger.info("Checking file: %s", filename)

        if result["rejected"] > 0:
            logger.warning(
                "%s | Rejected records: %s",
                filename,
                result["rejected"]
            )

        if filename in failed_files:
            logger.error(
                "%s | %s",
                filename,
                failed_files[filename]
            )
        else:
            logger.info(
                "%s | Total: %s | Valid: %s",
                filename,
                result["total"],
                result["valid"]
            )

    logger.info("Successful files: %s", len(successful_files))
    logger.info("Failed files: %s", len(failed_files))
    logger.info("Pipeline status: %s", pipeline_status)
    logger.info("Validation report completed")

finally:
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()

print("Log file created:", log_path)

Log file created: C:\tap-python-git\logs\day15_20261125_103642_354062.log


In [16]:
assert log_path.exists()
assert log_path.stat().st_size > 0

print(log_path.read_text(encoding="utf-8"))

print("Logging test PASSED")

2026-11-25 10:36:42,354 | INFO | Day 15 pipeline validation report started
2026-11-25 10:36:42,355 | INFO | Checking file: olist_customers_dataset.csv
2026-11-25 10:36:42,355 | INFO | olist_customers_dataset.csv | Total: 500 | Valid: 500
2026-11-25 10:36:42,355 | INFO | Checking file: olist_geolocation_dataset.csv
2026-11-25 10:36:42,355 | INFO | olist_geolocation_dataset.csv | Total: 2229 | Valid: 2229
2026-11-25 10:36:42,355 | INFO | Checking file: olist_orders_dataset.csv
2026-11-25 10:36:42,355 | INFO | olist_orders_dataset.csv | Total: 500 | Valid: 500
2026-11-25 10:36:42,355 | INFO | Checking file: olist_order_items_dataset.csv
2026-11-25 10:36:42,355 | INFO | olist_order_items_dataset.csv | Total: 576 | Valid: 576
2026-11-25 10:36:42,355 | INFO | Checking file: olist_order_payments_dataset.csv
2026-11-25 10:36:42,355 | INFO | olist_order_payments_dataset.csv | Total: 545 | Valid: 545
2026-11-25 10:36:42,355 | INFO | Checking file: olist_order_reviews_dataset.csv
2026-11-25 10:36

In [19]:
from pathlib import Path

project = Path(r"C:\tap-python-git")

for filename in [
    "day15_errors.py",
    "day15_logging.py",
    "day15_pipeline.py",
    "day14_pipeline.py",
    "day14_manifest.py",
    "reader.py"
]:
    print(filename, ":", (project / filename).exists())

day15_errors.py : False
day15_logging.py : True
day15_pipeline.py : True
day14_pipeline.py : True
day14_manifest.py : True
reader.py : True


In [20]:
from pathlib import Path

path = Path(r"C:\tap-python-git\day15_errors.py")

code = '''
import csv
from pathlib import Path


class SchemaError(Exception):
    pass


class FileFormatError(Exception):
    pass


class DataQualityError(Exception):
    pass


def check_source(path, required_fields):
    path = Path(path)

    if not path.is_file():
        raise FileFormatError(f"Missing file: {path.name}")

    try:
        with path.open(encoding="utf-8-sig", newline="") as file:
            reader = csv.DictReader(file)

            if not reader.fieldnames:
                raise FileFormatError("Empty file")

            missing = set(required_fields) - set(reader.fieldnames)

            if missing:
                raise SchemaError(f"Missing columns: {sorted(missing)}")

            if next(reader, None) is None:
                raise FileFormatError("No data records")

    except (OSError, UnicodeError, csv.Error) as error:
        raise FileFormatError(str(error)) from error


def check_quality(total, rejected, threshold):
    if total == 0:
        raise DataQualityError("No records available")

    if rejected == total:
        raise DataQualityError("No valid records available")

    rejection_rate = rejected / total

    if rejection_rate > threshold:
        raise DataQualityError(
            f"Rejection rate {rejection_rate:.2%} "
            f"exceeds threshold {threshold:.2%}"
        )
'''

if not path.exists():
    path.write_text(code.strip() + "\n", encoding="utf-8")
    print("day15_errors.py created successfully")
else:
    print("Module already exists")

day15_errors.py created successfully


In [21]:
import importlib

importlib.invalidate_caches()

from day15_errors import (
    SchemaError,
    FileFormatError,
    DataQualityError,
    check_source,
    check_quality
)

from day15_logging import setup_logger
from day15_pipeline import run_pipeline

print("All required imports successful")

All required imports successful


In [22]:
import json

first_run = run_pipeline(project / "config.yaml")

print(json.dumps(first_run, indent=4))

{
    "processed_files": 9,
    "skipped_files": 0,
    "failed_files": {},
    "total_records": 5739,
    "valid_records": 5739,
    "rejected_records": 0,
    "duration_seconds": 0.2004,
    "status": "SUCCESS",
    "log_file": "day15_20261125_104403_435705.log"
}


In [25]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "day15test.py",
        "-v"
    ],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Tests failed"

print("Automated testing completed successfully")

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\Syed Akil\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: C:\tap-python-git
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 7 items

day15test.py::test_valid_source PASSED                                   [ 14%]
day15test.py::test_missing_file PASSED                                   [ 28%]
day15test.py::test_invalid_schema PASSED                                 [ 42%]
day15test.py::test_rejection_threshold PASSED                            [ 57%]
day15test.py::test_threshold_boundary PASSED                             [ 71%]
day15test.py::test_column_mapping PASSED                                 [ 85%]
day15test.py::test_pipeline_idempotency PASSED                           [100%]

============================== 7 passed in 0.07s ==============================


Automated testin

In [26]:
from pathlib import Path

project = Path(r"C:\tap-python-git")
test_path = project / "test_day15.py"

code = r'''
import pytest

from day14_pipeline import validate_row
from day15_errors import (
    DataQualityError,
    FileFormatError,
    SchemaError,
    check_quality,
    check_source,
)


def test_valid_source(tmp_path):
    path = tmp_path / "valid.csv"
    path.write_text("order_id\n123\n", encoding="utf-8")
    check_source(path, ["order_id"])


def test_missing_file(tmp_path):
    with pytest.raises(FileFormatError):
        check_source(tmp_path / "missing.csv", ["order_id"])


def test_invalid_schema(tmp_path):
    path = tmp_path / "invalid.csv"
    path.write_text("customer_id\n123\n", encoding="utf-8")

    with pytest.raises(SchemaError):
        check_source(path, ["order_id"])


def test_rejection_threshold():
    with pytest.raises(DataQualityError):
        check_quality(100, 30, 0.20)


def test_threshold_boundary():
    check_quality(100, 20, 0.20)


def test_column_mapping():
    record = {"price": "100", "freight_value": "20"}

    config = {
        "thresholds": {"min_price": 0},
        "column_mappings": {
            "price": "item_price",
            "freight_value": "shipping_fee",
        },
    }

    result = validate_row(
        record,
        ["price", "freight_value"],
        config,
    )

    assert result["item_price"] == 100.0
    assert result["shipping_fee"] == 20.0
    assert "price" not in result
'''

if not test_path.exists():
    test_path.write_text(code.strip() + "\n", encoding="utf-8")

print("File created:", test_path.exists())
print("File size:", test_path.stat().st_size)
print("Location:", test_path)

File created: True
File size: 1415
Location: C:\tap-python-git\test_day15.py


In [27]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "test_day15.py", "-v", "--color=no"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\Syed Akil\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
rootdir: C:\tap-python-git
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 6 items

test_day15.py::test_valid_source PASSED                                  [ 16%]
test_day15.py::test_missing_file PASSED                                  [ 33%]
test_day15.py::test_invalid_schema PASSED                                [ 50%]
test_day15.py::test_rejection_threshold PASSED                           [ 66%]
test_day15.py::test_threshold_boundary PASSED                            [ 83%]
test_day15.py::test_column_mapping PASSED                                [100%]

============================== 6 passed in 0.09s ==============================


Exit code: 0


In [28]:
from pathlib import Path
from tempfile import TemporaryDirectory
import importlib
import yaml
import day15_pipeline

importlib.reload(day15_pipeline)

project = Path(r"C:\tap-python-git")

with (project / "config.yaml").open(encoding="utf-8") as file:
    demo_config = yaml.safe_load(file)

demo_workspace = TemporaryDirectory()
demo_project = Path(demo_workspace.name)

demo_data = demo_project / "data"
demo_data.mkdir()

new_filename = "reviewer_categories.csv"

new_file = demo_data / new_filename

new_file.write_text(
    "product_category_name,product_category_name_english,reviewer_note\n"
    "beleza_saude,health_beauty,Approved\n"
    "esporte_lazer,sports_leisure,Verified\n",
    encoding="utf-8"
)

print(new_file.read_text())

product_category_name,product_category_name_english,reviewer_note
beleza_saude,health_beauty,Approved
esporte_lazer,sports_leisure,Verified



In [29]:
demo_config["paths"] = {
    "data_dir": "data",
    "output_dir": "output",
    "log_dir": "logs"
}

demo_config["files"] = [new_filename]

demo_config.setdefault("required_fields", {})[new_filename] = [
    "product_category_name",
    "product_category_name_english",
    "reviewer_note"
]

demo_config["column_mappings"]["reviewer_note"] = "category_note"

demo_config_path = demo_project / "config.yaml"

demo_config_path.write_text(
    yaml.safe_dump(demo_config, sort_keys=False),
    encoding="utf-8"
)

print(demo_config_path.read_text())

paths:
  data_dir: data
  output_dir: output
  log_dir: logs
thresholds:
  max_rejection_rate: 0.2
  min_price: 0
column_mappings:
  price: item_price
  freight_value: shipping_fee
  order_purchase_timestamp: purchased_at
  reviewer_note: category_note
files:
- reviewer_categories.csv
required_fields:
  reviewer_categories.csv:
  - product_category_name
  - product_category_name_english
  - reviewer_note



In [30]:
import json

result = day15_pipeline.run_pipeline(demo_config_path)

print(json.dumps(result, indent=4))

{
    "processed_files": 1,
    "skipped_files": 0,
    "failed_files": {},
    "total_records": 2,
    "valid_records": 2,
    "rejected_records": 0,
    "duration_seconds": 0.0097,
    "status": "SUCCESS",
    "log_file": "day15_20261125_114404_682700.log"
}


In [31]:
import csv

output_file = (
    demo_project / "output" / "day15"
    / "clean_reviewer_categories.csv"
)

with output_file.open(encoding="utf-8", newline="") as file:
    records = list(csv.DictReader(file))

for record in records:
    print(record)

assert len(records) == 2
assert all("category_note" in row for row in records)
assert all("reviewer_note" not in row for row in records)

print("Schema evolution test PASSED")

{'product_category_name': 'beleza_saude', 'product_category_name_english': 'health_beauty', 'category_note': 'Approved'}
{'product_category_name': 'esporte_lazer', 'product_category_name_english': 'sports_leisure', 'category_note': 'Verified'}
Schema evolution test PASSED


In [32]:
second_run = day15_pipeline.run_pipeline(demo_config_path)

assert second_run["processed_files"] == 0
assert second_run["skipped_files"] == 1
assert not second_run["failed_files"]

print("Schema evolution and idempotency PASSED")

Schema evolution and idempotency PASSED
